## Import and Prepare Data 


In [12]:
import warnings 
warnings.filterwarnings("ignore", category=DeprecationWarning)

# PDF Loader 
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('economics_research_reference.pdf')
pages = loader.load() 

# Split Data 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_size=1800 , chunk_overlap=160)

split_data = spliter.split_documents(pages)

chunks = [i.page_content for i in split_data]
metadata = [i.metadata for i in split_data]

import hashlib 
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]

In [13]:
# Vector DB create 
import chromadb 
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction 
embedding_function  = DefaultEmbeddingFunction()

client = chromadb.PersistentClient(path="./VectorDB")

collection = client.get_or_create_collection(name="Self_RAG",embedding_function=embedding_function)

if collection.count()!=len(chunks):
    collection.add(ids=ids , documents=chunks,metadatas=metadata)

## Rank BM adding 

In [14]:
from rank_bm25 import BM25Okapi 
tokens = [i.lower().split() for i in chunks]
token_corpus = BM25Okapi(tokens)

### State connection 

In [15]:
from typing import TypedDict

class State(TypedDict):
    query: str
    context: list[str]
    answer: str
    needs_retrieval: bool
    grounded: bool
    retry_count: int

## Hybrid RAG adding 

In [16]:
def Hybrid_RAG(state:State) :
    query = state['query']
    result = collection.query(query_texts=[query],n_results=3)
    documents = result['documents'][0]
    distances = result['distances'][0] 
    print(distances)
    
    threshold = 1.0 
    dense_chunks = []
    for i , docs in zip(distances,documents):
        if i < threshold:
            dense_chunks.append(docs)
            
    scores = token_corpus.get_scores(query.lower().split())
    def get_index(score,k=10):
        index = list(enumerate(score))
        sorted_index = sorted(index,key=lambda x:x[1], reverse=True)
        return [doc for doc , i in sorted_index[:k]]

    keyword_corpus= get_index(score=scores,k=10)
    chunks_corpus = [chunks[i] for i in keyword_corpus]
    
    rrf_tokens = {}
    for rank , doc in enumerate(dense_chunks):
        rrf_tokens[doc] = rrf_tokens.get(doc,0.0)+1.0/(rank+60)
    for rank , doc in enumerate(chunks_corpus):
        rrf_tokens[doc] = rrf_tokens.get(doc,0.0)+1.0/(rank+60) 
        
    marge = sorted(rrf_tokens.items(),key=lambda x:x[1],reverse=True)
    
    top_docs = [i for i , doc in marge[:5]]
    content = "\n\n".join(top_docs)
    
    retry_count = state.get("retry_count", 0)
    return {
        "context": top_docs,
        "retry_count": retry_count + 1
    }


## LLM connection 

In [17]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq 
import os  

load_dotenv()
os.getenv("GROQ_API_KEY")

LLM = ChatGroq(model="openai/gpt-oss-120b")
LLM.invoke("hii").content


'Hey there! 👋 How can I help you today?'

### Connect LLM with state

In [18]:
def LLM_connection(state:State):
    prompt = f""" 
    context : {state['context']}
    question : {state['query']}
    """
    response = LLM.invoke(prompt)
    return {'answer':response.content}

### Grader 

In [ ]:

# Output structure 
from pydantic import BaseModel , Field 
class Grade(BaseModel):
    grounded : bool = Field(description="Whether the answer is supported by the context")
    
grader_llm = LLM.with_structured_output(
    Grade,
    method="json_schema"
)

def grader(state:State):
    prompt = f"""
    Check whether the answer is supported by the provided context.

    Context:
    {state['context']}

    Question:
    {state['query']}

    Answer:
    {state['answer']}

    Return only:
    YES
    or
    NO
    """
    
    response = grader_llm.invoke(prompt)
    
    # result = response.content.strip().upper()
    
    return {
        "grounded":response.grounded
    }
    
# Check 
def check_grounded(state: State):

    if state["grounded"]:
        return "end"

    else:
        if state["retry_count"] >= 2:
            return "end"

        else:
            return "retry"


### Graph Building 

In [20]:
from langgraph.graph import StateGraph, START, END 
builder = StateGraph(State)

builder.add_node("retrieval", Hybrid_RAG)
builder.add_node("llm", LLM_connection)
builder.add_node("grader", grader)

builder.add_edge(START, "retrieval")
builder.add_edge("retrieval", "llm")
builder.add_edge("llm", "grader")

builder.add_conditional_edges(
    "grader",
    check_grounded,
    {
        "end": END,
        "retry": "retrieval"
    }
)

graph = builder.compile()

In [21]:
# question = "What is the Full Form of EDA?"

# for event in graph.stream({
#     "query": question 
# }):
#     print(event)

In [22]:
question = "What is the Full form of LLM?"
result = graph.invoke({'query':question})
result['answer']

[1.5196640491485596, 1.6047828197479248, 1.8063995838165283]
[1.5196640491485596, 1.6047828197479248, 1.8063995838165283]


'The abbreviation **LLM** stands for **“Large Language Model.”**'